In [ ]:
import pandas as pd
from pathlib import Path

import rasterio
from rasterio.warp import calculate_default_transform
from rasterio.warp import reproject
from rasterio.warp import Resampling

In [ ]:
DIR_DATA = Path("data")
DIR_ICEYE_ORG = DIR_DATA / "1_org" / "1_ICEYE"
# FILEPATH_ICEYE_LIST = DIR_DATA / "2_processed" / "0_metadata" / "iceye-list.csv"
FILEPATH_ICEYE_LIST = DIR_DATA / "2_processed" / "0_metadata" / "iceye-list-test.csv"

TARGET_CRS = "EPSG:2958"
TARGET_RESOLUTION = 0.5  # metres

In [ ]:
images = pd.read_csv(FILEPATH_ICEYE_LIST)

for _, img in images.iterrows():
    image_path = DIR_ICEYE_ORG / img["filename"]

    with rasterio.open(image_path) as src:
        print("====================================")
        print(image_path.name)
        print(f"CRS      : {src.crs}")
        print(f"Size     : {src.width} x {src.height}")
        print(f"Bounds   : {src.bounds}")
        print(f"Transform:\n{src.transform}")
        print(f"Resolution: {src.res}")
        print("====================================")

        transform, width, height = calculate_default_transform(
            src.crs,
            TARGET_CRS,
            src.width,
            src.height,
            *src.bounds,
            resolution=TARGET_RESOLUTION
        )
        profile = src.profile.copy()
        profile.update(
            crs=TARGET_CRS,
            transform=transform,
            width=width,
            height=height
        )
        dst_path = DIR_ICEYE_ORG / (
            image_path.stem + "_EPSG2958_res05m.tif"
        )

        with rasterio.open(dst_path, "w", **profile) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    rasterio.band(src, i),
                    rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=TARGET_CRS,
                    resampling=Resampling.bilinear
                )
            print("====================================")
            print(dst_path.name)
            print(f"CRS      : {dst.crs}")
            print(f"Size     : {dst.width} x {dst.height}")
            print(f"Bounds   : {dst.bounds}")
            print(f"Transform:\n{dst.transform}")
            print(f"Resolution: {dst.res}")
            print("====================================")
